# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Helper bridge sudah disertakan di notebook ini. **Upload piano asli saja** saat diminta; tidak perlu mengunduh atau mengunggah ZIP tools.

Semua flag inferensi awalnya **False**. Konteks **90 detik sudah dipilih pendengar**; 120 detik dilaporkan banyak noise. Jalankan sel persiapan lalu bagian chunk seluruh lagu. Probe durasi bersifat opsional dan boleh dilewati. Kualitas sambungan lagu penuh belum divalidasi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

# Generated from repository scripts by scripts/build_context_bridge.py.
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "ace_context_chunks.py": {
    "sha256": "d7e4b17dea0647d859bbeea5d7809099c5582f7ee2b3bde05aec385864aa862c",
    "gzip_base64": "H4sIAAAAAAAC/6U827LbRnLv5ysg7QOAc4YgD3VZm+dAVV5Fu3ElsVW2k0oVw0INgSGJFQjAuJyLFVXlH/KH+ZL0ZQYYgCDlLetBIgYzPd09fZ+GXr58+d37D7OfG1U626LNE5XM4iJv1FPjqKdSVelR5U3tyDxxjkrWbaUS4dTyWGZqJrN0n6vEqZu0iQ9pvg+urn5Sba1qpzkoJz6o+BO83sq6nidVe3Qei+qTqurA+ZjJfF5WxVY5idrJNmucpnDywknznapUHqvg6gelktrJ22P5DDvGKf2DKO7STAlntzuWai+cMmt5O/WUAiL53nn/7//0nVO1eQO4O7uicmYz9aTitgGgL1++vEqPZVE1jqz2paxqZZ4Psj5k6dY8/r0ucvP7KJvD1a4qjk4Jv2CSo198xBf6d31omzTrntot0Berur4yQ0SKI4Gmsptl6MHhesd7EK1BDbyVmdkoLqpKZbJBwtM8iUolP/WQVabipkrjaNdmWQSI7wkeMCNTcCTf/fLBCZ1Xy8VicfXTjz/+Ag+IuBdFuHUU+UGl6iJ7UJ4fAEvwwNe3m6urKzgbIEsu37z1kHB/deXAnyTdq7oJNb8CPcGnd49pc2DgtCAoSpV7brV1fcKoqZQ8MhT8g4ezzYr4E5y7kzaq8jJ53CZyxRMBLZl4t4vl62v8yxdb1/VXvH3Qlgmww6PlvHelmrbKNXrBQT3xL8CMKXmsYIsIz5WQEw8ya5WmCQfCHu+7RiFnZfUc4nOAZEV1u9ulT54bNMfS5S27aQEDR73xcIcggcOuPdpCwIEBT8OlkFlWPEa5zMO/yqxW/o37X/kJpEqVmYwVI6JRz+Rz0TZeUzQyE45W0PDbhQDRrPZpHr4FuZCJCoM338D7VtXhD0UOwqJyVe2f6UFTSijVoaeBCAYgcDVjku4cT+bPXgryUzcSdNF7ENuiyHwHjisvGmfwxkvzRuyyQjZ+NwH1JUhrkFRgivfg00E/4CHz7n4nAXrB7eI+1Ajdh28XCzO+uEe87hlHHORf97f4Wy+YLa/N6MLvZauSaa2c/8D9PlRVUXkuGhSzCHb8v//5X9ipNix8F97WzETE01toxvjI7krB20XtdgwacYHPBRjh9wRdo9rdh/SKaKKBS/j9XLRVrJxjWzcOGEaDoVMrQDoxu//J+R4o2CvgRZGozEnaSjZpkdeAdpy1gL4CW13KJEGD7PygHmCqeoqReDSTlfoVTqBBVsgyIJBH+ZQe22NIxwZHCchoNvmEtNaQJKzQXHnMGHrj3xF3Qg0BjgKmGSbhm/vbr9Nt6DFHA3g5mZIP4EYysJ+AtcyBG7VTMExHtklaaHbEgFIT3uJ+mtUaFzCKsA88eEtBhMUqzTw+KsbTnyM0X8ORpYzTJgXVWOPwDUxAoClKw3ohaJ/Z7YbBEhoo1PS6kvleeTTD3/TUI2Lh7cqGrHHjSeCRYGixuXPKIktjmmDe0G8CAz9B1BzUYL03jJhNRnranNXT5nfoadOrZa+Zi/vmnng2JzeCNDdIM2JBcPDHi7AGN6QSr1aNR28uKuJfUIzA1DnooetO4BkN4bR5CiKKzh5hUugBdKQg2Si/6N302Y8O4JYPyd5aHSUAzfeshjPk+BoOsZvQgCirJjTjNyzg3bLr/uzWKbyf1+3RG46tNn7PtqxA6fM6cFr4Be8+XjxYekjDY5r3S0c7axAsZLPUv9agewAgDVnx7pCuJgxf4ewgdku3EGdsDe8x1qvGFoE1cK7tbZYe08bYHYtdiOghFUhqVggeHHBBIDX2W1A4wlbopTx6sxzRUJRkycI1n0PDr4dCx4Teh4Mp9+Eh3dis0JBWtkSjwhHu+qX4pJ5DDjicZiW3tdfMDDFaJ59DF6K4B5C9JDKcczuYqI6jHUBYU4xL6jCDaNRjwSSG3NwKRHU+By9FirOeZuNmEuUe8BBrjx086C5SzQ9sJhb+TbBYLK8HdM2H/Ca7runEyITXR7mSVcQr3DFYdysztChJhMrb8wIFN5AlhHsJCrHfGzUzyhv5V+PZJNl6+NDmnzorSOotPCm2Ph6+gvhZVRjz/ZaWpCiCtIVUsD8GMHhVIwAwSeBCyBmaehI8dtNbNO09D4z/DIGVyYxWM5u6GZC+KJ0EsBMxS/wuaOnZQBQY0j67GPo9uatUuOgyIgLvrqR+hDnuaitgVtk25iUToMdoBvwlBmdm/XEpCogMSu5qAs0vg/j4s8vpW4S8dFdkoFxiTbSrJBhkd8WMcjVxEdiiyAQhKxM3Xg2xIIvRz9JRpYvxVD+KT8I1ihQZCXFX5hfsSQyEbejfLzr+xUQgIrc/SERkI0VdhfWOMwWK6mX2KJ/raJmEv1TgRZLmuVShS47v1bKP4OrqRUgeDYQM4WAaUypIe16ES+P78rJ3jjjHDyCA974eY/ISkERIZQvIuz7982+DmMUkKgBS04d+GhJkyBKiYudB/J/rXfoXIY4GoJKe2w+6oh8tIYkDhRWfv/in01xMwl2/I79/x5ECRLw8RbiYqcPUCT/yc1uW7JPpdCwg9YqyfOHoxTaZ/SyTh1EVIAL19vp3hl4btTBknFYakk5op+bxvv3J6MQYhyOqM1gxATN/+OYPEctkYS5dE1XW+endPLffL8D80PA5eJQP5jdl764/pK2Th45ItsOey9yISkDFgOzqAIifBj0Y65AFYcFM36sp5xAOZJdgbn5faklVA51fXgv7HLSUMel8zOH0adMMvTnvzdk3D1kFCUHDPMUapvU8Ghw/JSkwnQsXrPVUC4qKTzrP5tlVCuSAlcPCRairFno/mrBHP0cWMyLjGzL8uce22L3hyQHXAQJwmKrytEpx8SeIi/IZuWP4OgbZmx/e/uT9i3CA5iVT86Oe6NBKkFKMNRLn8YDlJD5frIZhhIcCZPIlyKxyQ5mrKUJJ0cnle5kXeRoDWLJYmHqgH0YIGBHPj+kTQi3y7Dlw/sboQ+iOdSvaCY1Q9aASDc5QA2dNOHFJDCGAn4oPulKIsRKCmoHfhHALS4ok3DMUboe0puZUtS+rBVWbe2uXy4Ag57O8qBvIefEnyT1yicbxr9QVIHvmuGFAVjzCYemkd4VJMaxd4vp4JeGfMj5G9e3bTPFa4qW/EVToJMFjHhPnQstn8cQ+tUPd6fJEo1H9WR9VI9E5hFRJAs+V1B6pQTfXJ4fH5SZ/kAOYtewCmN6Ipc0F6RqK/TB4vZCoGahOUsCR6dQxPtDhWUdM9QuUWytnIIKHWCXFY75VElILsd74NyOUwVxhdE7vhlBOkkyOApSJUD2KXyyqqDgY0hmsORqkGTMOxL/xVxgVZir3aIrPb2/M283YZ1AQ4UFUUP9a0b9HOFauQF5fL32ND6pKqAt2FuyJeptAKgTj3q81lc0axOooowdV1RieLkHkZNPWkJGgLqrEFZ2p4FMdmg6hz56Mta3op8I+kBIjIiyzgqMKbcWDjz99+PnDL8Iy+f3PU7iWdbORGRu9IAeui9PJk1SdbALCCPT9oK8OKoX3FBB4zTshC5z3VmFJF8lq7d1qXV7gCtSgcvY9hO4YxWOMxNHoHbnHNG/T5plvRNqaTOWvrcxwrM3htNJdqpJAa4BVczY2l86YHTY5djtOwAHtoU/44e2KDFJ2YUUXVLXW4/M+EpzmO2+3du0hyJxGPhUvfWLQsvASOG1PhgD14GZQJC25whTR5UX6oKKm0Pj2+1I2bF8zvAgNFpf83986tIzlMS4QwKkaDlt7F3AUw4gUNzGFdTCuEZJh5xWjy4Ae0zvSbcsqE3knppjR4ZsDDkvmE6y32aTVs7+GGbHMAmVzS5vxF+HkQVxiXu/mfy/vAFPeRecZkChNhnZdwOpra0H4kjsPOL8coz3IPS9i/ZG2JoCMtUlxnSTdgfbXHIhY4c6pFlmsnNC/0UttRLWwkHI/R2QMbG0UNNJF/Jg20BmDxVZH/hWFo/TAmmsLlFFmC8qEr0e4oZ7Z7XHXhQ282CSBGLpiUthd7tiv2adgZGDUHBwLMtiepE+8OzAjAfYtztCJjzyTWfoH7MnkNlwxAeB0AOtBHWUzLOKcrO2qnn3tBAB5A0hYfdnMpoD7X4GuadjLlMAGfz6dMirc9FSMXgzYPFIkOPsx9edxHsDRBoQBDCUC5Mkc28Vax3vKiylgz7EkOAehwAv4em5wn6PLdo5pTdHi0AjjPqa60+asVLVXFQVrXmTZY1vVtG217HY3+atZZ2+a7mwtjc5op97qsiO+sqoNuzhvsv4a3OiyG/QgAgwWXXMvLm/4XhwHrVtxBIP3cPEnD/8SPPCvP77/l+jDf/639fTDX3rlgHPYg17q7NxsbUY54DiZPPBlNoCBzSHbP3hL+XXt+VyL+PxlcKNv6iXaX+iK3maYbGAVm3IoI6hYJ92AhyXja/Df8eroM097SdNeblaLV8kXd1jGpnW6FtCl/31OZqVIsDVhpylanYSUsYTQO+kDKz1xDQs343hpBBudOS8/deHkvVeTzhgXkPPpnC8R7J7uMfBBvJPtg07mlxUEt55LbTmOiU41dLxKELusrQ/MqDsd3qoBlD85P6m4wFtkSZXNGnGVDTYsNI7cNfgipxC6qtoSC2aGYRz9BsMLjQJrzxhgdkmwLTsGbprr6z/Nt2CfFVtQGH4dXQ9MU5+mkTvVk2wfOnVU2uilLNIgyxjRW/qg39uKMPSZoe0zV5MlhMFpabzs47rr2aHf3m1hw09jke2mmavg0+20Z2DnUT43B0g5KIZFkwo6DOe4k2nmbNUOL69j2Ia6peDscN9nw/rTBKs9bsGiLk5ecKVJHxCoqjmcz7yCldTvbMWKh2/C21OF67hwGdgpCpyWmmNg96Frbx1QQf7vfIwwXf3p74X+oZBA6FazCdNzqr4dhuc0eGB62F6aNcFpSgV+ze5xsi22ME/DcFcPaj+cbD27J2pQ9VgurvMSNGN/u/CwlMKjPF3cqtkt1T+0Pz/WuuoxAWhcPqHnFmvRtERf2MAoTX/72u/hlthmOIB8LPKC6zsMTIKshbf+erX6ZhPIGkF5FigTAmNZBpf6929fU7ALCPMzUPJ61V2UHX5zV1TndrETskjSGLJ8d7UIvnS7z0L8m3fX1b847Lr1CKrQfyV4D4UlfgGB4KFI4GkH0eu6wwa7CTSIOXJAd4t4t6KfAsNS3yqbMZ0UZMWjOKT7g74Zf71YLObLNwu++rTGXr3xCZ6MYUNddsJuQoqCTGshvF0DwBUCRK+MYzcQdT3aWStCoTeUk8KKxeY+XPxu9m2xi5CFAtYSoA3SDIb2LpP7EB0X4l5yBZp8Nk6ixhdYsHkXIojr4Ns3pomnyHdpgu2jYSdscZaWRI3cb+aEo1iIW390JYqoEndgGhtcA+ldGLx5w3HOBC39PHNVSTeejwr4BkbPbkiRaJ/xWDOISEvs8FsIeCrT+ZIbV05F38YRSSngXBAKCLhAZDYIAAz/aNR0bip5jB7hSItHEJZdIypESmRkZ0TF/8QKHbd4TJMu3Nbmkl7M6MV8vrzDG3V6c0NDTFKIcNc0PMumjCLeXU+/YUnfhoSUBlGdBVGdB4HSwLe3kL4RbmJJArk9GZ240/wrNvyBd4dNapDI5wKCgG0BgZiu15rLzEcpHrfh4HCZZ3fbDFkjrx/lzfb6cctVNCkUkIZ20Bf4zxZTjh132JH585S8VvJGba/V1p8vjfxWVS+5bNEmzNg1rKD1W47NUUloR/8d2K8/98KqrzbBjYVg2wkPRNaH0Gjncwy0s5YsAn1r8xHGIaz7tU0V3Qlqa5Y4DZieOuXuGVBGyPDT35TmF6aAWcblGAyltm2aJa1uMQTRbUKPCMa+FG5LIcz8Gw+fvdsZ7uPrxsEKAx20MX3f22KUPuo2D0Ekic8uC6zVqEDP7JRdlNMIGBAl2x28A24oaV37uCSFowlbn9o02M9iawU+CJf260N7XuOuiBgLYlkpMBJYX8jUA4RkXGCjqQR8jpYOTk37TtyqbtwVnz7+9o1F4fb6SFaVfK4pPxXk8mqtr7prZpht3ZF0al/Arwa9GBur9Up7RF7qvwi7W4t6SmM+QAzzrJM807pHFaG8ce17WTRkv6kKbJY3VecDdRwZvFfLvq2PjZO+gswd6vgh9IakG/2nCMBo+++M17SVGDd6cHgx7vS4VAFhLGn3eY7+gLo/RlUPMpdia6JQqxtoI+wxwrev1BAn13K13ehrLDlJy2o7O28g0Q8Mm6rsnknr5C16e3/BGOsuRPN7c5cXRyzwafM9oKdvxVIS7LbpGgYjdRu80TW+Ub/QZv7KH/UCWq103lrvtumr0KdNRBtCMbzYsWf/IaPCXNDgZ4yw0I83/Ng17i0X1lUrZsWNBD5KjIPARjsHme34uxd93WS61tuma16l9nJYoSBxd1Syt5LjpkqPA3Uds4j4YzzFiEPrmLWGmiS7YbKlnQd/Z85qICI3uO8kf9CGxzfs5Tog/j27/IF2IYhB9yVXQQwa0x2pKJVcJ6N+CjjCvCGWOElax7JKLMWhSBEFyLNjGrYELJjm90ZYkU6sw5op3vh2H+XT6mn9arNmE7wZKKx2L5jz1xRuWhLwPbhe7FYQOoDQpw8eI9ZXhm4N59pSby0VSFLsZC1oXq4eSRyCsbIPhXElteYzoaO3M/swjC2QU6MnNmW7Gok58arfbLPezqYEZnLZ5MxpM0b8tE4WqxEDCk3b5gQR+uL+lf81Gqks9XVSpod1A8CrgUhf6sgauomsDKsyPJv98BDeWrsr9yN6dvtWWUc9knrLZvuqAPkyH965X4adx1kpYCNOipmPvtApMpJvGQpqjLpdLhbXFIBCFr/0qnKNKG7mmf5BoZd5IO03U0YBpc4Rmtp0KlBVNkyN3TQmK9TPHIJx9IEc5hAglDyEwRQPbM80uVLsRoSFSDJFavxYlUywCa6YUPr7DKyjzFvAsFIPqXoMPSJjfSak2zj3zuwVpbbb2kw9H9Zt/Hdvz3XpDv6gMaKPKu0QF/mN29AbiOMxcT9DAjcdaBoivMdKASlufJtewj0SP/MXhTtIcWsnKWhvhRUc7OBiUZs36XFbqcD5gC0NsxJb3Pqen32LpYdGQbx3kLWy5DawjDWFG6YDmrjWFxms5Aa0HGsZQLJu7tPOFa/MQg4WBGT2c1zGNxDwgyXxNrAzAV5+jesgC6Dd3RX9I1ysGmPniJIJ0Hfk+7gVTcWTpGcqeLgr/Ptsl7VjFFbzMEnlHvvN0rimVrg7/HZVtk0B+pXGTpzJ9EifKx2KCs41jttKxs/CYfYi42pqeYuf6V5w2EYSuKPIHzvd/sid2PCyiy74sOVm8tJr6mbIm75NmriYZosLseaduWqw486vXgwBoae3LhO3RJO3MDruABjTDSeXL5OmMh19wWlfy/T9EN09/KCkC+8ni7nMGKMVVmsg3cD6HbvMDALE7+yUyhwEC4ZKXJO/6VbYrtY+0c+sl+ip/Ll0fdd1bDppY+4GJPDxAIrPu7jW0Qp9xXI+JT1tzNWXqTv+RNa0Qe3cz50gfuG+NN6BnEXdbrlv/+P7f4uWr91pGO4xfaKmaV4fvLn2gj9f28zlRpMb3YJ3ATSKJ+JDX/t1iIluB1tK/2AP6pABTLvftZhm6fZYvoK4VOGSLY3dfrv8NL0WproTDahEDcbXmOfRia21YbTI4PqITjnw7XpcQtmMUrLTM6Rw/DMtHqiTdaA6ntPlxFcMctx4yW9v9NvNpZPqDp+VmcmrwRjZ0taXEHpFMnWEofERRvPGnxDpz4e8iYT9wvWP9RWcyvfNgTzZN8EQzKAmYQMew5joHhxSjZIwYc10e+HAm33mpfrLGmH3eUSGB+A3rb6QkTdEh0kRB36BqSO5vth2SszU3VoPwyxkNgm7KYiJc1e2B4JTPN8T9cX/w7qJX75Qgzh9/2U1jJ8ywp01/I5R/4dV97wTitj+Rmxdzym3vpc2XcK6KbjzCNPNuV3SItQTc5HKcKGlgxOxT9dbFk6Ya0uATM/wWdvun+2G5WiR5ZUpG5R5eUiHQ/hVrinKlaH5/zuC76o90fgRnyovAV9dpfSJZxhFSRFHkX8HchFCtJkkEUoITaxxZhOCzB4hK0hweyuStnq5YQWt5GWey+2F1AtK41Lv77mzGXscV5Dhwuv3EdTpVQW26J1b0jeGj5eZ/wDBKBODoBBb6P9WJfx2cWZPLi9dXPv2wt6Yu11cHLz55szO2PVukXthE/sLNgil0xhLXcPv1Dbdhvrjo87+o/SRO3fpAxJYgcGb0JoycOjjI8alIDMn+NDJXz4oE4kChBdGK8GsnoDCTgUblKEC/3+Wqa11WwE4Vfo8AkGjqQD2KPdkZy6CAs0TG3d9hbw5/q8AHWvfLsS3C3G7XGzGYnsKh6381MnrixqYXIPOEUtxZe11tw34FGi9C/mDBtf+ar9GMOYTNZqsezroN2iL0BD4+1Qthjw4LJvymH3zoZfilxeWpaTB8fdpKpvAlflqNbxdyMNobZ9b9X1qfX8bwYs+00xzNJyR3P2eljYuvfS9OWd7cmjiSUsOjdodOZr3X+vs7hpuCG/qurncJikuYqQ7dPik6WKv674ZMKZrrqFR/XD+rFDfh/dE1rKVJtVqPu2JmTg4KrlFgleJ6Pwp00QN3ErgR7O4L8/6r4I0YPN/BVHTC6AckReNIiAnitABRhF+XY2O8Or/AW5E1kW9SwAA"
  },
  "electric_full_song.py": {
    "sha256": "04d99be5e6bf1635b7ac16347680f344be13344fa1b8953523c9543a96c502ce",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwXVyr58lJikRTMGpurAjc61gtkC3/GWE65CK9ojEGYB5lkohzY6v379aqRDJ5jxyVVkpV/4bJiI4FwjJy6wyViLFFjktsrBqu7AHgk4+FVaMIIOUKVoefIkjB25dg6R4lX19z6F+S8EUXs/Ut7ktLYsO95MK0478LfP22Q3YZ3B8jJY9XudOqgJXsFG5WlCpzDOZ7eL5tlDp32j7JNZKSTXeXfuv/v8i396cvqOVB58/+vZp/r7w/X7X05O/3Z2+VZ/nV5dnl/8+vbio2PZ72nh6c1ucWmMdFqbc9tBc6C3zjBE7+oDXJ2fv7+4PJs6x7DPp48nlzfnVx8/nH28sXu0vmvA0N0T1cYjB5hwaYRiPcCuzEvU+na4tW++1rAm857m4dI2EanBw+QOBrorN+K5aPfp57cnYp3Ubgcm8yjyrTFHyu/0+vPow/WNmAdxPAvCe6dxFp0x8cCKi2KgWIz0Aj/rT3skzoWrFXYb7IMzmKsPtK+Bme1zTsK/42Vca0nvJJQIF91lXRElkk+Z9RVIO58FLVKyX6d43ELGc5BNQb4A4Xt4eP+Av7oZLm5nDtb1AsS9XDIw6QWnXYGiRYn2hAqBoHaa8Qk8iRdh7QRQKWNQlx40hmU5Htu5vTY1CnfC3uit+jRCP3v028nZtC5dmMfrYml5Ga3YdrHOKM/6LLjagGoCG8DjbwLp0y00aBlaWpSb/brX241RgEYT+824iCYjGGUICuf/wr8tCjKVP7rHbS/QJhwfgwq+/yICqcBnJu0EU+PobOZTAhlQz18+gq4xnAKL7BDsPwX+4Y7bqm4EJR7TntXWmQUE3lK9/an6vp1aE2lHEl/N9m4sTpNCCxr7QdcGFN6OvnQFRYcAYEAL8TTyL+IUpBf6JdpkccEj26QKfERTOvWdXbDFfgDVax2xqca5FU9rORRXnp0Hgn07aSFjZpD93nA2TSNbIgud00+bIZa6uT1Mx236IjVg7shcaf2rz5nFQeKaLbUmss2xIVYytacbtYjpjd/TmTFYMcFheNyvbJCpqaRCriZBMjVEyNVYOzww4I1GLZQOQTXLqaaYMINVpk5Bcs9Hued0XCYLigA86yLanq1btKUExAYwapWruUadT/erc016Uw3MWmKysVvddjdABNfygPBN1BygWG24p9ZHy1YN7YMWEDX0gIKmNTCGFQVOaxLoQ2q1Whv6WMqGomSdg6CnOE4+tXw67tH4KPNtLT00+RmOYz5vowEjQlmJ5ZnyjH4iAtEkQJ1qrdUhTHCAynUxdeZgvAAnDXnGdO48kjKiL5TQaPP5/hN4stTUKPJBJc62KEqk2Dr7NxKWcYvQuGE52MkNh+kaWY6Swcd2SuhFSZ7/jxxilQ8MQIyupuigsm88faTDPw1bicL94VJr3T+cHFTzdloQbDA6xD7n6YqpMwI/nEpAkPPyjdSFvEFYVkqIV+cVHYM+kAP3wUJqx/gPowj9mibSycNxq0pVP8bKxoSiLsO6gNWHc26UfPBW2WtEoeus1BcqBTEj8bsxyDII/g8Io2LyvVnmBmZjNYODxQBTWmnGrcf/9Oq+veZunIOUAriibOZfVLhTuE2nUReEaMhW9TKgkop1XHpfVeYgBCvYopPeAXgTDw1oD/bc1t5vLmWEuqC7o42oXtxZyKMghq6E9v5NZedYqlSBiu/rfKGgIQeBMctYjb+49t+enb8HxnvLlTZga6tNy4ybU21qDHJswx73oYYz2rpksE3MRA751vubacPql+GUJoImdqTItbV1VatqVNCoEWqyTJ1RR742bmGPLBgOfYuMx2P/hHDuY7H2eKTJwgzVSmuk/9axSFPdRAqlwmxTDDiO8zcpMxGIYgUaQ2DUGXUvuc8MDhHMMa9xdfVhhEvB+vWzgFVQUCE/VumEwRo0exWOaickdtBYo7YNgwhkBRbuc9SEY/8UIjIhmz7ctys4GyRsFU7uJDUmdRqtsaGxKnVwsV1pcbir2GJoLH1tHWvi7FkEs0ZB5oLQmmGl30RXJltFyMDXfRMHfyzkdnr14cPFp5vh3kqPb4i5sWYio9XXKsCaAp7IYiGjxg5VEBAL0iO0BG2q0jkBC1msYZGMLtPyHHRlZAprdOoZp82xHawsnm7MLLZ6aSWr8NOYuI23G8yFVhG5mWiVnWMq0qp7sAdhxsvHSIiuC+Sw7j4zofE2yERsIjWfY9U9P2wBg6GgQ4IDZxIutSLSOjbiCOafXaM/ade+mwih2XbQxZF9PSp+Nynfxhsm/Z5BPz/iWHqab7+tNkALPcr3kUmOeSWW87CXeZBDySbLfWNdgHHrCT1W8iKQqvjDZbx11K9jHDMfM+kK52mGuiUfDDN+LY+qk/B51sPq7spRMXyOVdTurs65YQUv3gufcmGlpkrW6ZoicvWTLkoeRjIDIxpZuX7OhdRgXnKJ307OqHCkVBSEBfVB7ogn3gV59AAoAAelJOI372dgg4Ksl3CLT8JAEyUS8/xcIldKQrTn7En29KhnRmWPJ2el5/pL3hpBjaFoFY7apbW4Q9uFwwBFQhPJLJg2U0/fegOLc7QqmPQEVufO9fuTywm9vXs0G99+10zbfXf3VBicUkz81+vPonrd9Xy4lY910N756vJMXJ/c3Ezsnes9J97r+ZMunB4+czgdtmcKrV8OVjS244xW0Q5a593KfV0KTirRJ2Hj+5ZmJIfiiNN/ZPEf4SynTmjwAMBZw/4f1CFJIhUKj7Hgk/ncJ3cMA6w13VSKrC6kAT043Zlh6roVNfndOnxgE/6gEGbP7qYCTWvPTnjNUkRuX9iuY8sOGkrKJlJrJKUctCTllIMxgSIHc4v4gqQefuuA9i6wXA5GHr+4/j0lkYU+PVXDW/kEJiSVbNKQY4/1OWEnDCp29r4d31nPmui5Un8cs2OXOxZm0ECnKKqxSapXUfB7SIHpBsj0DJOsJXVMRv6OnA2OgJVuW9UAdziiYq1253MAtTMx+kC8Mzs4aDOouWJIasfCAiiHsqkUBPQROby4BJxCliUVptzdOjwI9VtfL0+1QvzSoBgVcv2alLWqlRPQ71ixJmDY94B11/PV/koau76hgmWr9AG3LMH/oUA+6s1YLtLmK9sgbryG7WwDBjX4WT4/F1VfzZZkjeDqIOnQvbKawMUChi79GhLc07++GY13KHpWwQc462AhTdeTTZD4AIKRRSR4L7dEffR2gcxFaBnqTzAYDaI8he/63MGzj1dOWLc3HobQfsDDeaEN1x67le2skSm8ajARk4+WOfSsmc6MD7zw/ZrVO8PKA79QXzVTHbcHIIxyEMzpingbRumUEkHquct9TirCteKtuNCIdhYV0VvKi733nZK438vvl8Fk4aEaxB+ELfpBLxR5Nl2ShgHN3oLARfDBpwEP5QPg1vgsDTQoJjwxN2qxJnrEjdUaUl73sJBvDNN81JFo7ecwfZOI13hOIwfrmFJ3ulF7JSTlUAvSvSuEPRdhmwcvQmBkGMImg5c/z6pVE9bwsE4a6dyDaGYcDGCbdfJNod4uKLvrPCRqV6pVcSqrQG6wV1O8fMlG6XRr0UZh292zALsgJYPP5Bb8ugW9O1ubWznEga0i6vRE9b640i7opKKhhwK7Pn5fJLU+d6NM7m7HVvsX6+ltxGd3bVa/q3vuhhiD7WzWirE/f6V+F6a20YaVqJp2FTam8IiKfU3FU/33rmizpYN1rs62cjq9zs6wdSaDe37+DbyIth0YWbMtOIqNBXeOesnCOs2yf93moN3LEhkyI0w7vNSlHzD4EZO7JjRj+7v2ZHrDQkzrqSbMLHOQ7qbtzdCQF9PG9HmS+VbvtZWIOtjtVb4wi9pPu/+LDKoOcO+y99sOkC2x9dRGDKvpnvEneWeWgjfzetX4t0Iaox27Av+WVCEgVwDGMl2QseIv4iOQgOIXN1VmseLNEUt+tmikmez9wSz0i66j39/DWUxEnkpb0B8zZS7eSb4gW/uaetxIcgQS4+a+H6Uh+PfWTPwnHfxAT3Fr/x4pZHpNOQXYMgBGn+6KEeh//KK4Pb7bu3L1T15Ua+8dzjEF6xwmLGr5/Lt2ohpqPZnMmfoW470zrZdv9mT896KmztVKla1nfyZKtgq+qNV6ha92jAW7YwcdqqJaGkIKKOQU87hwqf0zqyhMz0R9xIoIbj5fo0d40yiF0evSX7iyybmib4PFkLyBXZPRqP6mMUgeduKEGgFV7X8RRDZ8ArNDUb1k48dc+kSSrR1DIthvkO20FTPco0oZVSca2svXhzK5I+5sxiy5sS6kN9kkatYfg1ZEz7khiTqpOPLWWMbo+/6H4LoE7DYnQJZVWHzIMpacB5/rMX30pYmTD/4HNKQt7d5MAAA="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
# Explicitly load this notebook's helper, even if an earlier notebook imported one.
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
embedded_single = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embedded_single)
sys.modules["electric_full_song"] = embedded_single
spec = importlib.util.spec_from_file_location("ace_context_embedded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; notebook ini belum menyediakan model/GPU")

print("Upload satu file piano asli saja; tidak perlu ZIP tools")
uploaded = files.upload()
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(audios) == 1, "Pilih satu file piano asli"
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=90, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Opsional: `90` sudah dipilih, sehingga tidak perlu diulang. `60` tersedia sebagai kontrol; `120` hanya untuk eksperimen eksplisit meskipun sudah dilaporkan berisik. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, context.generation_source(OUT, plan), PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 90 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=60, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
